# Download Planet Basemap Scenes for ARTS Using Orders API
This script downloads the 2018 July-Sept. basemap grids which contain delineated RTS.

In [1]:
import os
import json
import requests
import urllib.request
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely as shp
from pprint import pprint
import math
import time
import ast

In [2]:
# Get Planet API Key
%load_ext dotenv
%dotenv

api_key = os.getenv('PL_BM_API_KEY')

## Define Functions

In [3]:
def makemydir(dir_path):
    try:
        os.makedirs(dir_path)
    except OSError:
        pass

In [4]:
def recu_down(url, filename): # recurrent download with ContentTooShortError
    try:
        urllib.request.urlretrieve(url,filename)
    except urllib.error.ContentTooShortError:
        print('Download failed. Trying again...')
        recu_down(url, filename)

# Import Data

In [5]:
grids_filtered = gpd.read_file('../data/new_planet_grids_for_arts_closest_year_v.3.1.0.geojson')

In [6]:
grids_filtered

,planet_basemap_year,id,link,replicate,geometry
0,2017.0,312-1641,https://api.planet.com/basemaps/v1/mosaics/a2b...,1,"POLYGON ((-1840755.368 -325221.87, -1835119.93..."
1,2017.0,313-1644,https://api.planet.com/basemaps/v1/mosaics/a2b...,1,"POLYGON ((-1822903.507 -327838.148, -1817322.6..."
2,2017.0,313-1669,https://api.planet.com/basemaps/v1/mosaics/a2b...,1,"POLYGON ((-1688385.062 -303645.821, -1683215.6..."
3,2017.0,313-1670,https://api.planet.com/basemaps/v1/mosaics/a2b...,1,"POLYGON ((-1683215.652 -302716.134, -1678062.0..."
4,2017.0,314-1644,https://api.planet.com/basemaps/v1/mosaics/a2b...,1,"POLYGON ((-1821889.135 -333429.195, -1816311.4..."
...,...,...,...,...,...
437,2016.0,734-3308,https://api.planet.com/basemaps/v1/mosaics/e95...,1,"POLYGON ((-1692145.698 -602532.98, -1689553.37..."
438,2016.0,734-3310,https://api.planet.com/basemaps/v1/mosaics/e95...,1,"POLYGON ((-1686965.01 -600688.26, -1684380.61 ..."
439,2016.0,735-3306,https://api.planet.com/basemaps/v1/mosaics/e95...,1,"POLYGON ((-1696413.17 -606986.338, -1693814.31..."
440,2016.0,735-3307,https://api.planet.com/basemaps/v1/mosaics/e95...,1,"POLYGON ((-1693814.313 -606056.452, -1691219.4..."


In [7]:
print(grids_filtered.planet_basemap_year.sort_values().unique())
grids_filtered_annual = [
    grids_filtered[grids_filtered.planet_basemap_year == year] 
    for year 
    in grids_filtered.planet_basemap_year.sort_values().unique()
]
print([len(df.index) for df in grids_filtered_annual])

n = 5000
grids_filtered_annual_chunks = [
    [annual_grids[i:i+n] for i in range(0,annual_grids.shape[0],n)]
    for annual_grids 
    in grids_filtered_annual
    ]
pprint([[len(df.index) for df in year] for year in grids_filtered_annual_chunks])

[2016. 2017.]
[314, 128]
[[314], [128]]


In [29]:
makemydir('../data/download_20250715/')
order_info_path = '../data/download_20250715/planet_basemap_orders.csv'
order_download_path = '../data/download_20250715/planet_basemap_downloads.csv'


for annual_grids in grids_filtered_annual_chunks[1][0:1]:
    
    basemap_name = 'global_quarterly_' + str(int(annual_grids.planet_basemap_year.iloc[0])) + 'q3_mosaic'

    requested_list = []

    # split image info into list of chunks of approximately 5 basemaps to download
    n = 5
    grids_filtered_list = [annual_grids[i:i+n] for i in range(0,annual_grids.shape[0],n)]
    
    count = 0
    
    # submit order for each item in the list
    for chunk in grids_filtered_list:
        count += 1
        
        # Try this section
        if os.path.exists(order_info_path):
            prior_orders = pd.read_csv(order_info_path,
                                       converters = {'order_info': ast.literal_eval})
            prior_orders = [row['order_info'] for idx, row in prior_orders.iterrows()]
            prior_names = [item['name'] for item in prior_orders]

        else:
            prior_names = []

    # Get chunk info
        item_ids = list(chunk.id)

        order_name = (
            basemap_name
            + '_'
            + item_ids[0]
            + '_'
            + item_ids[len(item_ids)-1]
        )

        print("-------------------------------------")
        print(count, ': ', order_name)

        if len(item_ids) > 0: # if images to download

            if order_name not in prior_names: # check if an order has already been placed

                print(len(item_ids), "items to be ordered:")
                print(item_ids)

                # create the order info
                order_info = {
                    "name": order_name,
                    "source_type": "basemaps",
                    "products": [{
                        "mosaic_name": basemap_name,
                        "quad_ids": item_ids
                    }]
                }


                # Place order
                print("Requesting items...")
                request = requests.post('https://api.planet.com/compute/ops/orders/v2', 
                                        auth=(api_key, ''),
                                        json=order_info)
                print(str(request))
                if str(request) in ['<Response [400]>', '<Response [401]>']:
                    sys.exit("Invalid request or credentials. Check your request.")
                elif str(request) != '<Response [202]>':
                    time.sleep(5)
                    print("Requesting items...")
                    request = requests.post('https://api.planet.com/compute/ops/orders/v2', 
                                            auth=(api_key, ''),
                                            json=order_info)

                

-------------------------------------
1 :  global_quarterly_2017q3_mosaic_312-1641_314-1644
5 items to be ordered:
['312-1641', '313-1644', '313-1669', '313-1670', '314-1644']
Requesting items...
<Response [202]>
-------------------------------------
2 :  global_quarterly_2017q3_mosaic_314-1669_317-1619
5 items to be ordered:
['314-1669', '315-1643', '315-1661', '316-1620', '317-1619']
Requesting items...
<Response [202]>
-------------------------------------
3 :  global_quarterly_2017q3_mosaic_317-1620_319-1623
5 items to be ordered:
['317-1620', '317-1625', '318-1624', '319-1622', '319-1623']
Requesting items...
<Response [202]>
-------------------------------------
4 :  global_quarterly_2017q3_mosaic_319-1624_329-1672
5 items to be ordered:
['319-1624', '320-1621', '320-1622', '320-1623', '329-1672']
Requesting items...
<Response [202]>
-------------------------------------
5 :  global_quarterly_2017q3_mosaic_330-1673_332-1671
5 items to be ordered:
['330-1673', '330-1674', '331-167

In [30]:
orders = requests.get('https://api.planet.com/compute/ops/orders/v2',
                      params = {'source_type':'basemaps'},
                      auth=(api_key, '')).json()
order_links = {order['name']:order['_links']['_self'] for order in orders['orders']}

try:
    next_page = orders['_links']['next']
except:
    next_page = None
    
while next_page:
    orders = requests.get(next_page,
                          params = {'source_type':'basemaps'},
                          auth=(api_key, '')).json()
    new_links = {order['name']:order['_links']['_self'] for order in orders['orders']}
    order_links.update(new_links)
    
    try:
        next_page = orders['_links']['next']
    except:
        next_page = None
        
for annual_grids in grids_filtered_annual_chunks[1][0:1]:
    
    basemap_name = 'global_quarterly_' + str(int(annual_grids.planet_basemap_year.iloc[0])) + 'q3_mosaic'

    requested_list = []

    # split image info into list of chunks of approximately 5 basemaps to download
    n = 5
    grids_filtered_list = [annual_grids[i:i+n] for i in range(0,annual_grids.shape[0],n)]
    
    count = 0
    
    # get chunk info
    for chunk in grids_filtered_list:
        count += 1
        
        # Try this section
        if os.path.exists(order_info_path):
            prior_orders = pd.read_csv(order_info_path,
                                       converters = {'order_info': ast.literal_eval})
            prior_orders = [row['order_info'] for idx, row in prior_orders.iterrows()]
            prior_names = [item['name'] for item in prior_orders]

        else:
            prior_names = []

    # Get chunk info
        item_ids = list(chunk.id)

        order_name = (
            basemap_name
            + '_'
            + item_ids[0]
            + '_'
            + item_ids[len(item_ids)-1]
        )

        print("-------------------------------------")
        print(count, ': ', order_name)

        if len(item_ids) > 0: # if images to download

            if order_name not in prior_names: # check if the order has already completed and been recorded

                # create the order info
                order_info = {
                    "name": order_name,
                    "source_type": "basemaps",
                    "products": [{
                        "mosaic_name": basemap_name,
                        "quad_ids": item_ids
                    }]
                }
                
                order_link = order_links[order_name]
        
                try:
                    request = requests.get(order_link, auth=(api_key, '')).json()
                except:
                    time.sleep(5)
                    request = requests.get(order_link, auth=(api_key, '')).json()

                # wait while the order is queued and runs
                while request['state'] == 'queued':
                    time.sleep(1)
                    try:
                        request = requests.get(order_link, 
                                                    auth=(api_key, '')).json()
                    except:
                        time.sleep(5)
                        request = requests.get(order_link, 
                                                    auth=(api_key, '')).json()

                while request['state'] == 'running':
                    time.sleep(1)
                    try:
                        request = requests.get(order_link, 
                                                    auth=(api_key, '')).json()
                    except:
                        time.sleep(5)
                        request = requests.get(order_link, 
                                                    auth=(api_key, '')).json()

                # If the order succeeded, create and append order info into file
                if request['state'] == 'success':

                    print('Order succeeded.')
                    print("-------------------------------------")
                    print("\n")

                    requested_list.append([order_info, request])

                    order_df = pd.DataFrame({
                        'order_name': order_name,
                        'order_info': [request]
                    })

                    order_df.to_csv(order_info_path,
                                    index = False,
                                    mode = 'a',
                                    header = not os.path.exists(order_info_path))

                # If the order failed, try again
                elif request['state'] == 'failed':
                    # Place order
                    print("Trying order again...")
                    try:
                        request = requests.post('https://api.planet.com/compute/ops/orders/v2', 
                                                auth=(api_key, ''),
                                                json=order_info)
                    except:
                        time.sleep(5)
                        request = requests.post('https://api.planet.com/compute/ops/orders/v2', 
                                                auth=(api_key, ''),
                                                json=order_info)

                    # wait while the order is queued and runs
                    try:
                        request = requests.get(request.json()['_links']['_self'], 
                                               auth=(api_key, '')).json()
                    except:
                        time.sleep(5)
                        request = requests.get(request.json()['_links']['_self'], 
                                               auth=(api_key, '')).json()

                    while request['state'] == 'queued':
                        time.sleep(1)
                        try:
                            request = requests.get(request['_links']['_self'], 
                                                   auth=(api_key, '')).json()
                        except:
                            time.sleep(5)
                            request = requests.get(request['_links']['_self'], 
                                                   auth=(api_key, '')).json()

                    while request['state'] == 'running':
                        time.sleep(1)
                        try:
                            request = requests.get(request['_links']['_self'], 
                                                   auth=(api_key, '')).json()
                        except:
                            time.sleep(5)
                            request = requests.get(request['_links']['_self'], 
                                                   auth=(api_key, '')).json()

                    # If the order succeeded, create and append order info into file
                    if request['state'] == 'success':

                        print('Order succeeded.')
                        print("-------------------------------------")
                        print("\n")

                        requested_list.append([order_info, request])

                        order_df = pd.DataFrame({
                            'order_name': order_name,
                            'order_info': [request]
                        })

                        order_df.to_csv(order_info_path,
                                        index = False,
                                        mode = 'a',
                                        header = not os.path.exists(order_info_path))

                    # If the order failed, stop the code from running any farther
                    elif request['state'] == 'failed':
                        print('last_message:', request['last_message'], '\nerror_hints:', request['error_hints'])
                        break

            else:
                print('Order already placed.')
                print("-------------------------------------")
                print("\n")


-------------------------------------
1 :  global_quarterly_2017q3_mosaic_312-1641_314-1644
Order succeeded.
-------------------------------------


-------------------------------------
2 :  global_quarterly_2017q3_mosaic_314-1669_317-1619
Order succeeded.
-------------------------------------


-------------------------------------
3 :  global_quarterly_2017q3_mosaic_317-1620_319-1623
Order succeeded.
-------------------------------------


-------------------------------------
4 :  global_quarterly_2017q3_mosaic_319-1624_329-1672
Order succeeded.
-------------------------------------


-------------------------------------
5 :  global_quarterly_2017q3_mosaic_330-1673_332-1671
Order succeeded.
-------------------------------------


-------------------------------------
6 :  global_quarterly_2017q3_mosaic_332-1673_333-1675
Order succeeded.
-------------------------------------


-------------------------------------
7 :  global_quarterly_2017q3_mosaic_335-1672_337-1631
Order succeede

In [16]:
# # In case I need to cancel orders quickly
# requests.post('https://api.planet.com/compute/ops/bulk/orders/v2/cancel',
#               auth=(api_key, ''))

<Response [400]>